# Grundlegende LLM-Ansteuerung mit Python und Ollama

Dieser Code bietet ein Beispiel davon, wie man mithilfe von Ollama direkt ein LLM via Python ansteuern kann. Dieser Code lässt sich direkt für z.B. Zusammenfassungen von Texten nutzen. Für strukturierte Outputs empfehle ich das Notebook "llm_beispiel_strukturiert".

## Voraussetzungen

Dieser Code setzt einige Software und grundlegende Python-Skills voraus. Installationshilfen für Software sind im README zu finden. Bei den notwendigen Python-Skills kann man sich relativ einfach von LLMs helfen lassen - es handelt sich um sehr grundlegende Aufgaben, die die meisten LLMs sauber lösen können.

### Software

- Python (möglichst aktuelle Version)
- pip (sollte gemeinsam mit Python installiert werden)
- Ollama

### Python-Pakete

Notwendig für die grundlegende Ansteuerung sind:
- pandas
- numpy
- ollama
- re
- pathlib (nicht direkt notwendig, aber macht Dateipfade einfacher)

Manuelle Installation ist möglich; bei Forschungsprojekten empfehle ich eine Installation via requirements.txt (siehe README).

In [1]:
import pandas as pd
import numpy as np
import ollama
import re
from pathlib import Path

## Vorbereitung: Text einlesen

Es gibt verschiedene Varianten, um Dateien in Python einzulesen. Ich nutze eine persönliche Funktion, die CSVs direkt einliest, aber ich lege euch nahe, eine gute Lösung für eure spezifischen Daten zu finden. Wichtig ist, dass ihr die Texte, die ihr an das LLM geben wollt, letztendlich als List speichert.

In diesem Beispiel schreibe ich einfach eine manuelle Liste von drei kurzen Texten, aus denen ich das LLM anweisen werde, diese danach zu bewerten, ob sie Beispiele für "negative campaigning" sind oder nicht.

In [2]:
texts = [
    "Andy Burnham will bring new energy and ideas to the Labour Party. He has a proven track record of leadership and is committed to addressing the issues that matter most to the British people.",
    "Keir Starmer has been unable to solve the problems affecting Britain. He is lacking imagination and leadership, and it is time for him to make way for new faces.",
    "The government has been right in highlighting climate change as an important issue, but its Net Zero goal and the policies associated with it have been overly ambitious and caused significant economic harm."
]

## Prompting

Prompts werden einfach als Text gespeichert. Ich empfehle, die Instruktionen jeweils als Systemprompt zu formulieren. Das erleichtert meiner Meinung nach den Workflow, und verhindert Interferenzen zwischen Anweisungen und Text. Ich nutze hier einen sehr generischen Prompt; in tatsächlicher Anwendung würde man diesen iterativ entwickeln. Ein nützlicher Trick für lange Prompts kann es sein, einzelne Teile separat zu speichern (vor allem, wenn diese ein bestimmtes Format haben), und sie mit f-strings einzubinden - ein Beispiel dazu findet sich in "llm-beispiel-strukturiert"

Systemprompts werden oft strukturiert geschrieben, mit Markdown oder ähnlichen Formatierungen als Hilfe. Ich zeige hier einen einfachen strukturierten Prompt mit Role Assignment.

In [3]:
sysprompt = '''You are a political scientist with a deep knowledge of political campaigning, but no ideological preferences. You are tasked with determining whether a given statement made by a British political party contains an example of negative campaigning or not.

## Definition of Negative Campaigning

Negative campaigning refers to any instance where a political candidate or party attacks their opponent's character, record, or policies rather than promoting their own platform. Negative campaigning does not have to be uncivil or aimed at the character of the opponent, and also includes factual criticism of an opponent's policy.

## Instructions

You will be given a text statement made by a British political party. Determine whether the statement contains an example of negative campaigning or not. If it does, respond with "Yes". If it does not, respond with "No". Add a short explanation for your decision.
'''

## Setup und Inferenz

Ollama verlangt drei Setup-Schritte:
1) **Modelle herunterladen:** Falls ein Modell zum ersten Mal genutzt wird, muss es erst heruntergeladen ("pulled") werden. Dies geschieht mit dem ollama.pull() Befehl. Wird ein bereits heruntergeladenes Modell erneut gepullt, wird es nicht neu heruntergeladen; man kann also den Pull-Befehl problemlos jedes Mal ausführen. Wir benutzen hier das Mistral 7B-Modell - ein kleines, lokal ausführbares Modell aus Europa.
2) **Client laden:** Um das LLM anzusteuern, benötigen wir einen Client. Diesen initialisieren wir mit ollama.Client().
3) **Optionen und Messages festlegen:** "Messages" sind die Prompts, die wir weitergeben wollen. Wir können dabei zwei Rollen nutzen, "system" und "user". Typischerweise nutzen wir "system" für die Anweisungen, und "user" für die zu klassifizierenden Texte. Optionen werden in einem Dictionary angegeben. Typische Optionen wären "seed" und "temperature" (siehe Präsentation), aber weitere Optionen können oft auf der Ollama-Modellseite ([Modellseite für Mistral](https://ollama.com/library/mistral-small3.2)) eingesehen werden.

Die eigentliche Inferenz geschieht mit dem client.chat()-Befehl. In diesem Befehl spezifizieren wir das Modell ("model"), die Prompts ("messages"), und die Optionen ("options"). Theoretisch könnten wir uns einfach die Resultate ausgeben lassen, es lohnt sich jedoch, diese in einer Variable zu speichern und weiter zu bearbeiten.

Hier benutzen wir sehr wenig post-processing. Wir geben lediglich die kompletten, rohen Ergebnisse aus (print(out), etwas, was wir eher selten machen), und die kompletten Textausgaben.

In [10]:
modelname = "mistral:7b"

ollama.pull(modelname)

client = ollama.Client()

out = [] #Leere Liste, um die Ergebnisse zu speichern

opts = {
    "seed": 42,
    "temperature": 0.7, #Wir benutzen hier eine eher hohe Temperatur; oft würde man auch 0.0 für Reproduzierbarkeit nehmen - siehe Präsentation
}

for text in texts:

    messages = [
        {"role": "system", "content": sysprompt},
        {"role": "user", "content": text}
    ]

    response = client.chat(
        model=modelname, 
        messages=messages, 
        options = opts)

    out.append(response) #Füge die Antwort der Liste hinzu

#Was hat Mistral produziert? Rohe Ergebnisse ausgeben
print(out)

#Post-processing
output = [response.message.content for response in out]
print(output)

[ChatResponse(model='mistral:7b', created_at='2026-07-29T12:43:38.9615172Z', done=True, done_reason='stop', total_duration=14140176100, load_duration=39936600, prompt_eval_count=230, prompt_eval_duration=1077724000, eval_count=59, eval_duration=13016660000, message=Message(role='assistant', content=' No. The statement does not contain an example of negative campaigning as it focuses on highlighting positive attributes of Andy Burnham (new energy, ideas, leadership skills, commitment) and his potential contributions to the Labour Party and British people, without directly criticizing any other candidate or party.', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None), ChatResponse(model='mistral:7b', created_at='2026-07-29T12:43:54.4609794Z', done=True, done_reason='stop', total_duration=15498194500, load_duration=50499900, prompt_eval_count=226, prompt_eval_duration=1036742000, eval_count=68, eval_duration=14394817000, message=Message(role='assistant', content='